# Extension 01 Geometric constraints

In [ ]:
%pip install -q numpy session-py==0.310.0 session-jupyter==0.2.0

# Question

**How does a support slide along an inclined rail until only a perpendicular reaction remains?**

```
n42 ─ n43 ─ n44 ─ n45 ─ n46 ─ n47 ─ n48
 │     │     │     │     │     │     │
n35 ─ n36 ─ n37 ─ n38 ─ n39 ─ n40 ─ n41
 │     │     │     │     │     │     │
n28 ─ n29 ─ n30 ─ n31 ─ n32 ─ n33 ─ n34
 │     │     │     │     │     │     │
n21 ─ n22 ─ n23 ─ n24 ─ n25 ─ n26 ─ n27
 │     │     │     │     │     │     │
n14 ─ n15 ─ n16 ─ n17 ─ n18 ─ n19 ─ n20
 │     │     │     │     │     │     │
n7  ─ n8  ─ n9  ─ n10 ─ n11 ─ n12 ─ n13
 │     │     │     │     │     │     │
n0  ─ n1  ─ n2  ─ n3  ─ n4  ─ n5  ─ n6
```

**Given:** the $6 \times 6$ net of 49 nodes $n_0 \ldots n_{48}$ and 84 bars $e_0 \ldots e_{83}$, two inclined rails on the left ($x = 0$) and right ($x = 6$) edges, force densities $q$, and downward load ${\color{#2e9e4a} p_z = -0.15 \text{ kN}}$ on every free node.

**Unknown:** the positions of the ${\color{#d62828} \text{free}}$ interior nodes and the ${\color{#2a6fdb} \text{sliding}}$ rail supports.

**Equilibrium:** the force residual across all nodes is

$$ r = {\color{#2e9e4a} p} - C^{T}\,Q\,C\,x $$

At ${\color{#d62828} \text{free}}$ nodes $r_N = 0$. At ${\color{#2a6fdb} \text{fixed}}$ supports $-r_F$ is the reaction force. When a support slides without friction along a rail with unit tangent $t$, projecting $r_i$ onto $t$ moves the support along the rail until the tangential force vanishes:

$$ r_{\parallel,i} = (r_i \cdot t)\,t, \qquad {\color{#2a6fdb} n_i} \leftarrow {\color{#2a6fdb} n_i} + \alpha\,r_{\parallel,i} $$

<div align="center">
  <img src="https://raw.githubusercontent.com/BRG-teaching/Teaching/main/CAS/logos/fd_projections.svg"
       style="width:100%; height:auto;"
       alt="Projection of bar forces and support residuals into tangent and normal components" />
  <p style="text-align:center;">
    <em>Projecting the support residual onto the rail tangent drives slip to zero so only a perpendicular reaction remains.</em>
  </p>
</div>

**References:**

- **Constrained Form Finding Using the Force Density Method**, https://github.com/BlockResearchGroup/compas_fd
- **Another Structural Analysis Package**, https://github.com/keithjlee/Asap.jl

<hr style="border: none; border-top: 2px solid #555; margin: 2em 0;">

# Network

In [1]:
import numpy as np

# nodes: 6 x 6 cells = 7 x 7 nodes n0..n48
n = 7
x, y = np.meshgrid(range(n), range(n))
x = x.flatten().astype(float)
y = y.flatten().astype(float)
z = np.zeros(n * n)

# bars and connectivity matrix C
edges = [(k, k+1) for k in range(n*n) if x[k] < n-1] + \
        [(k, k+n) for k in range(n*n) if y[k] < n-1]
C = np.zeros((len(edges), n * n))
for e, (a, b) in enumerate(edges):
    C[e, a], C[e, b] = 1, -1

# four pinned corners and ten sliding nodes on the left and right rails
corners    = [0, n-1, n*(n-1), n*n - 1]
left_rail  = [k for k in range(n*n) if x[k] == 0   and 0 < y[k] < n-1]
right_rail = [k for k in range(n*n) if x[k] == n-1 and 0 < y[k] < n-1]
fixed = sorted(corners + left_rail + right_rail)
free  = [k for k in range(n*n) if k not in fixed]

C_N, C_F = C[:, free], C[:, fixed]

# left rail rises from z = 0 to 2.4 m, right rail falls from z = 2.4 to 0 m
a_L, b_L = np.array([0.0, 0.0, 0.0]), np.array([0.0, 6.0, 2.4])
a_R, b_R = np.array([6.0, 0.0, 2.4]), np.array([6.0, 6.0, 0.0])
for k in [0] + left_rail + [n*(n-1)]:
    z[k] = 2.4 * (y[k] / 6.0)
for k in [n-1] + right_rail + [n*n - 1]:
    z[k] = 2.4 * (1.0 - y[k] / 6.0)

# Iteration

At every step we solve `# equation 6.32` for the free nodes, evaluate the nodal residuals $r$, and slide the side supports along their rail tangents until $\max \|r_{\parallel}\| < 10^{-6}\text{ kN}$.

In [2]:
# stiffer cables on the bottom and top edges (q = 5), interior q = 1.5
q = 1.5 * np.ones(len(edges))
for e, (a, b) in enumerate(edges):
    if (y[a] == 0 and y[b] == 0) or (y[a] == n-1 and y[b] == n-1):
        q[e] = 5.0
Q = np.diag(q)

# downward load on the free nodes
p_x = np.zeros(n * n)
p_y = np.zeros(n * n)
p_z = np.zeros(n * n)
p_z[free] = -0.15

t_L = (b_L - a_L) / np.linalg.norm(b_L - a_L)
t_R = (b_R - a_R) / np.linalg.norm(b_R - a_R)

for step in range(200):
    # equation 6.32
    A = C_N.T @ Q @ C_N
    x[free] = np.linalg.solve(A, p_x[free] - C_N.T @ Q @ C_F @ x[fixed])
    y[free] = np.linalg.solve(A, p_y[free] - C_N.T @ Q @ C_F @ y[fixed])
    z[free] = np.linalg.solve(A, p_z[free] - C_N.T @ Q @ C_F @ z[fixed])

    # nodal residuals across all nodes
    r_x = p_x - (C.T @ Q @ C) @ x
    r_y = p_y - (C.T @ Q @ C) @ y
    r_z = p_z - (C.T @ Q @ C) @ z

    # slide rail supports along their unit tangents
    max_tan = 0.0
    for sliders, a_pt, b_pt, t in [(left_rail, a_L, b_L, t_L), (right_rail, a_R, b_R, t_R)]:
        for k in sliders:
            r_k = np.array([r_x[k], r_y[k], r_z[k]])
            r_tan = np.dot(r_k, t) * t
            max_tan = max(max_tan, np.linalg.norm(r_tan))
            pt = np.array([x[k], y[k], z[k]]) + 0.25 * r_tan
            s = np.clip(np.dot(pt - a_pt, b_pt - a_pt) / np.dot(b_pt - a_pt, b_pt - a_pt), 0.0, 1.0)
            x[k], y[k], z[k] = a_pt + s * (b_pt - a_pt)

    if max_tan < 1e-6:
        break

print("steps =", step + 1)
print("max free residual   =", round(np.abs(r_z[free]).max(), 12), "kN")
print("max rail slip force =", f"{max_tan:.2e}", "kN")

steps = 41
max free residual   = 0.0 kN
max rail slip force = 9.44e-07 kN


# Viewer

The inclined rails and sliding supports are blue, the free nodes are red, and the support reactions $-r_i$ are green.

In [3]:
from session_py import Arrowhead, Color, Line, Matrix, Session
from session_jupyter import show

# the colours of the figures: free nodes red, fixed nodes blue, loads green
RED   = Color(0.84, 0.16, 0.16, 1.0)
BLUE  = Color(0.16, 0.44, 0.86, 1.0)
GREEN = Color(0.18, 0.62, 0.29, 1.0)

f = q * np.sqrt((C @ x)**2 + (C @ y)**2 + (C @ z)**2)
colors = [Color(0.65 - 0.65 * fv / f.max(), 0.65 - 0.65 * fv / f.max(), 0.65 - 0.65 * fv / f.max(), 1.0) for fv in f]

session = Session("fdm_ext01_line_rails")
session.add_network(Matrix.from_rows(C.tolist()), Matrix.from_rows(np.column_stack([x, y, z]).tolist()), colors, [3.0] * len(edges))

for k in range(n * n):
    point = session.objects.points[k]
    point.pointcolor = BLUE if k in fixed else RED
    point.width = 13.0 if k in fixed else 7.0

for a_pt, b_pt in [(a_L, b_L), (a_R, b_R)]:
    rail = Line(a_pt[0], a_pt[1], a_pt[2], b_pt[0], b_pt[1], b_pt[2])
    rail.linecolor = BLUE
    rail.width = 4.0
    session.add_line(rail)

for k in left_rail + right_rail:
    arrow = Line(x[k], y[k], z[k], x[k] - 0.5 * r_x[k], y[k] - 0.5 * r_y[k], z[k] - 0.5 * r_z[k])
    arrow.linecolor = GREEN
    arrow.width = 3.0
    arrow.arrowhead = Arrowhead.END
    session.add_line(arrow)

show(session, "fdm_ext01_line_rails")

# Observation

Every green reaction vector stands perpendicular to its inclined blue rail. When a roller is aligned with $x$, $y$, or $z$, no loop is needed at all: each axis can use its own `free` and `fixed` lists in one linear solve.